# 02B - Trade-Finance / Contingent Facilities LGD

This notebook adds a dedicated trade-finance / contingent-facilities LGD segment under the parent commercial cash-flow framework.

Product examples:
- bank guarantees
- contingent obligations
- trade-related short-term facilities


In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.data_generation import generate_commercial_data
from src.commercial_data_controls import assign_framework_segment
from src.lgd_calculation import exposure_weighted_average
from src.reproducibility import set_global_seed

set_global_seed(62)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 140)
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT_DIR = os.path.abspath('../outputs')
TABLE_DIR = os.path.join(OUTPUT_DIR, 'tables')
FIG_DIR = os.path.join(OUTPUT_DIR, 'figures')
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

SHOW_PLOTS = os.environ.get('LGD_NOTEBOOK_SHOW_PLOTS', '0') == '1'
print(f"Interactive plot display enabled: {SHOW_PLOTS}")


## Governance Baseline (Pre-Step-3)

- **Fallback hierarchy:** observed workout inputs -> approved proxies -> conservative fallback with disclosure.
- **Proxy logic:** contingent conversion, claim probability, and cash-security strength are synthetic but transparent.
- **Discount-rate policy:** `discount_rate = max(contract_rate_proxy, cost_of_funds_proxy)`.
- **Calibration status:** this is a portfolio-project proxy baseline, not calibrated to internal guarantee-claim history.


## Objective
Build an interview-ready trade-finance / contingent-facilities LGD segment focused on contingent conversion to EAD, security/cash support, and recovery timing.

## Drivers
- product type and transaction type
- contingent vs already drawn status
- claim/crystallisation probability
- tenor
- cash security and collateral support
- customer risk strength

## Logic
- EAD is driven by contingent conversion plus already drawn balances.
- LGD is driven by security/cash backing, customer strength, legal/processing costs, and recovery timing.

## Downturn
- Stress increases claim conversion and conversion-to-funded EAD, weakens security benefit, and lengthens recovery timing.

## Validation
- Bounds checks on contingent amounts, claim probability, EAD floor vs drawn, and recovery-time sanity.

## Limitations
- Synthetic proxy assumptions; production use requires internal claim and recovery evidence.


## Why This Segment Differs from Standard Term Loans

- Exposure may be **contingent** and crystallise into funded EAD when claims are called.
- Tenor is often shorter and transaction-linked.
- Some facilities are fully or partially **cash-secured**.
- Workout path can be faster/structurally different than longer-term amortising loans.


In [ ]:
# Build trade / contingent segment on the same base dataset as parent commercial notebook
all_loans, _ = generate_commercial_data(n_loans=420, seed=43)
all_loans = all_loans.copy()

all_loans['icr'] = pd.to_numeric(all_loans['interest_coverage_ratio'], errors='coerce')
all_loans['industry_risk_bucket'] = all_loans['industry_risk_level'].fillna('Medium')
all_loans['watchlist_flag'] = (
    all_loans['default_trigger'].isin(['Covenant Breach', 'Voluntary Administration', 'Receivership'])
    | (all_loans['icr'] < 1.35)
).astype(int)

# Parent framework mapping alignment (must match parent notebook)
all_loans['framework_segment'] = assign_framework_segment(all_loans)

trade = all_loans[all_loans['framework_segment'] == 'Trade / Contingent Facilities (Proxy)'].copy()

print('All loans:', len(all_loans))
print('Trade/contingent segment loans:', len(trade))
display(trade[['loan_id', 'facility_type', 'security_type', 'ead']].head(10))


In [ ]:
# Trade/contingent product and risk structure proxies
rng = np.random.default_rng(62)

trade['product_type'] = rng.choice(
    ['Bank Guarantee', 'Performance Bond', 'Standby LC', 'Trade Contingent Facility'],
    size=len(trade),
    p=[0.40, 0.25, 0.20, 0.15],
)

trade['transaction_type_proxy'] = rng.choice(
    ['Domestic Trade', 'Import / Export', 'Construction Contract', 'General Corporate Obligation'],
    size=len(trade),
    p=[0.35, 0.25, 0.25, 0.15],
)

trade['is_contingent_flag'] = np.where(trade['product_type'].isin(['Bank Guarantee', 'Performance Bond', 'Standby LC']), 1, 0)
trade['tenor_months'] = rng.integers(3, 25, size=len(trade))

industry_risk_factor = trade['industry_risk_bucket'].map({'Low': 0.0, 'Medium': 0.5, 'Elevated': 1.0}).fillna(0.5)

trade['cash_security_pct'] = (
    0.28
    + 0.42 * (trade['product_type'] == 'Bank Guarantee').astype(int)
    + 0.12 * (trade['product_type'] == 'Standby LC').astype(int)
    - 0.15 * trade['watchlist_flag']
    + rng.normal(0.0, 0.10, len(trade))
).clip(0.0, 1.0)

trade['collateral_support_pct'] = (
    0.20
    + 0.50 * trade['security_coverage_ratio'].clip(0.0, 1.4)
    + 0.08 * (1 - industry_risk_factor)
    + rng.normal(0.0, 0.08, len(trade))
).clip(0.0, 1.0)

trade['security_level'] = np.select(
    [trade['cash_security_pct'] >= 0.95, trade['cash_security_pct'] >= 0.40],
    ['Fully Cash-Secured', 'Partially Secured'],
    default='Unsecured / Weakly Secured',
)

trade['customer_risk_strength'] = (
    0.60
    + 0.18 * ((trade['icr'] - 1.20) / 2.00).clip(-0.6, 1.0)
    - 0.20 * trade['watchlist_flag']
    - 0.12 * industry_risk_factor
    + rng.normal(0.0, 0.08, len(trade))
).clip(0.05, 1.00)

# Contingent and drawn profile
trade['facility_limit_proxy'] = trade['facility_limit'].clip(lower=trade['drawn_balance'])

trade['already_drawn_amount'] = (
    trade['drawn_balance']
    * np.where(trade['is_contingent_flag'] == 1, rng.uniform(0.10, 0.55, len(trade)), rng.uniform(0.60, 1.00, len(trade)))
).clip(lower=0.0)
trade['already_drawn_amount'] = np.minimum(trade['already_drawn_amount'], trade['facility_limit_proxy'])

trade['undrawn_headroom_amount'] = (trade['facility_limit_proxy'] - trade['already_drawn_amount']).clip(lower=0.0)

trade['contingent_commitment_amount'] = (
    trade['undrawn_headroom_amount']
    * np.where(trade['is_contingent_flag'] == 1, rng.uniform(0.75, 1.00, len(trade)), rng.uniform(0.20, 0.70, len(trade)))
).clip(lower=0.0)

trade['claim_probability_base'] = (
    0.08
    + 0.18 * trade['watchlist_flag']
    + 0.12 * (1.0 - trade['customer_risk_strength'])
    + 0.10 * industry_risk_factor
    + 0.06 * ((trade['tenor_months'] - 6) / 18.0).clip(0.0, 1.0)
    + 0.05 * (trade['is_contingent_flag'])
    + rng.normal(0.0, 0.03, len(trade))
).clip(0.01, 0.95)

display(
    trade[
        [
            'product_type', 'is_contingent_flag', 'tenor_months',
            'cash_security_pct', 'collateral_support_pct', 'security_level',
            'customer_risk_strength', 'facility_limit_proxy',
            'already_drawn_amount', 'undrawn_headroom_amount', 'contingent_commitment_amount',
            'claim_probability_base',
        ]
    ].head(12)
)


In [ ]:
# EAD conversion logic (base and downturn)
trade['conversion_factor_base'] = (
    trade['claim_probability_base']
    * (1.0 + 0.05 * trade['watchlist_flag'] + 0.04 * ((trade['tenor_months'] - 6) / 18.0).clip(0.0, 1.0))
).clip(0.01, 1.00)

trade['ead_from_contingent_base'] = trade['contingent_commitment_amount'] * trade['conversion_factor_base']
trade['ead_base'] = trade['already_drawn_amount'] + trade['ead_from_contingent_base']
trade['ead_base'] = np.maximum(trade['ead_base'], trade['already_drawn_amount'])
trade['ead_base'] = np.minimum(trade['ead_base'], trade['facility_limit_proxy'])

trade['claim_probability_downturn'] = (
    trade['claim_probability_base']
    + 0.08
    + 0.10 * trade['watchlist_flag']
    + 0.06 * (1.0 - trade['customer_risk_strength'])
).clip(0.02, 1.00)

trade['conversion_factor_downturn'] = (
    trade['claim_probability_downturn']
    * (1.08 + 0.08 * trade['watchlist_flag'])
).clip(0.02, 1.00)

trade['ead_from_contingent_downturn'] = trade['contingent_commitment_amount'] * trade['conversion_factor_downturn']
trade['ead_downturn'] = trade['already_drawn_amount'] * (1.0 + 0.03 * trade['watchlist_flag']) + trade['ead_from_contingent_downturn']
trade['ead_downturn'] = np.maximum(trade['ead_downturn'], trade['already_drawn_amount'])
trade['ead_downturn'] = np.minimum(trade['ead_downturn'], trade['facility_limit_proxy'])

trade['ead_conversion_uplift_pct'] = (
    (trade['ead_base'] - trade['already_drawn_amount']) / trade['already_drawn_amount'].replace(0, np.nan)
).fillna(0.0).clip(0.0, 5.0)

ead_conversion_summary = (
    trade.groupby('security_level', observed=True)
    .apply(
        lambda g: pd.Series(
            {
                'loan_count': len(g),
                'total_contingent_commitment': g['contingent_commitment_amount'].sum(),
                'total_already_drawn': g['already_drawn_amount'].sum(),
                'total_ead_base': g['ead_base'].sum(),
                'total_ead_downturn': g['ead_downturn'].sum(),
                'ead_weighted_claim_prob_base': exposure_weighted_average(g, 'claim_probability_base', 'ead_base'),
                'ead_weighted_claim_prob_downturn': exposure_weighted_average(g, 'claim_probability_downturn', 'ead_base'),
                'ead_weighted_conversion_uplift_pct': exposure_weighted_average(g, 'ead_conversion_uplift_pct', 'ead_base'),
            }
        ),
        include_groups=False,
    )
    .reset_index()
    .sort_values('total_ead_base', ascending=False)
)

print('=== EAD Conversion Summary ===')
display(ead_conversion_summary.round(4))


In [ ]:
# LGD logic (base/downturn/final)
security_level_factor = trade['security_level'].map(
    {'Fully Cash-Secured': 0.0, 'Partially Secured': 0.6, 'Unsecured / Weakly Secured': 1.2}
).fillna(0.6)

trade['post_claim_recovery_months'] = (
    4
    + 10 * (1.0 - trade['cash_security_pct'])
    + 6 * trade['watchlist_flag']
    + 3 * security_level_factor
    + np.random.default_rng(620).normal(0.0, 1.8, len(trade))
).clip(2.0, 36.0)

trade['legal_processing_cost_pct'] = (
    0.010
    + 0.040 * (1.0 - trade['cash_security_pct'])
    + 0.020 * trade['watchlist_flag']
    + 0.010 * security_level_factor
).clip(0.005, 0.18)

trade['lgd_base_economic'] = (
    0.38 * trade['realised_lgd']
    + 0.27 * (1.0 - trade['cash_security_pct'])
    + 0.14 * (1.0 - trade['collateral_support_pct'])
    + 0.11 * (1.0 - trade['customer_risk_strength'])
    + 0.04 * trade['watchlist_flag']
    + 0.04 * ((trade['post_claim_recovery_months'] - 6.0) / 18.0).clip(0.0, 1.0)
    + trade['legal_processing_cost_pct']
).clip(0.05, 0.92)

trade['post_claim_recovery_months_downturn'] = (
    trade['post_claim_recovery_months']
    * (1.12 + 0.10 * trade['watchlist_flag'] + 0.06 * security_level_factor)
).clip(2.0, 48.0)

trade['downturn_scalar'] = (
    1.00
    + 0.08 * trade['claim_probability_downturn']
    + 0.10 * (1.0 - trade['cash_security_pct'])
    + 0.05 * ((trade['post_claim_recovery_months_downturn'] - trade['post_claim_recovery_months']) / 24.0).clip(0.0, 1.0)
).clip(1.00, 1.45)

trade['lgd_downturn'] = (trade['lgd_base_economic'] * trade['downturn_scalar']).clip(0.0, 1.0)
trade['moc_addon'] = 0.015 + 0.010 * trade['watchlist_flag']
trade['supervisory_floor_proxy'] = trade['security_level'].map(
    {'Fully Cash-Secured': 0.03, 'Partially Secured': 0.18, 'Unsecured / Weakly Secured': 0.35}
).fillna(0.25)
trade['lgd_final_regulatory'] = np.maximum(trade['lgd_downturn'] + trade['moc_addon'], trade['supervisory_floor_proxy']).clip(0.0, 1.0)


def weighted_lgd(df, group_col):
    rows = []
    for k, g in df.groupby(group_col, observed=True):
        rows.append(
            {
                'dimension': group_col,
                'bucket': str(k),
                'loan_count': len(g),
                'total_ead_base': g['ead_base'].sum(),
                'ead_weighted_lgd_base': exposure_weighted_average(g, 'lgd_base_economic', 'ead_base'),
                'ead_weighted_lgd_downturn': exposure_weighted_average(g, 'lgd_downturn', 'ead_base'),
                'ead_weighted_lgd_final': exposure_weighted_average(g, 'lgd_final_regulatory', 'ead_base'),
            }
        )
    return pd.DataFrame(rows)

weighted_by_product = weighted_lgd(trade, 'product_type')
weighted_by_security = weighted_lgd(trade, 'security_level')
segment_summary = pd.concat([weighted_by_product, weighted_by_security], ignore_index=True)

base_vs_downturn = pd.DataFrame(
    [
        {
            'ead_weighted_lgd_base': exposure_weighted_average(trade, 'lgd_base_economic', 'ead_base'),
            'ead_weighted_lgd_downturn': exposure_weighted_average(trade, 'lgd_downturn', 'ead_base'),
            'ead_weighted_lgd_final': exposure_weighted_average(trade, 'lgd_final_regulatory', 'ead_base'),
        }
    ]
)
base_vs_downturn['downturn_sensitivity_pp'] = (
    (base_vs_downturn['ead_weighted_lgd_downturn'] - base_vs_downturn['ead_weighted_lgd_base']) * 100
)
base_vs_downturn['final_minus_base_pp'] = (
    (base_vs_downturn['ead_weighted_lgd_final'] - base_vs_downturn['ead_weighted_lgd_base']) * 100
)

print('=== Weighted LGD by Product Type ===')
display(weighted_by_product.round(4))
print('=== Weighted LGD by Security Level ===')
display(weighted_by_security.round(4))
print('=== Base vs Downturn Comparison ===')
display(base_vs_downturn.round(4))


In [ ]:
# Sensitivity analysis: claim conversion and security support

def run_trade_sensitivity(df, claim_add=0.0, cash_sec_shift=0.0):
    x = df.copy()
    claim = (x['claim_probability_base'] + claim_add).clip(0.01, 1.0)
    cash_sec = (x['cash_security_pct'] + cash_sec_shift).clip(0.0, 1.0)

    ead_s = x['already_drawn_amount'] + x['contingent_commitment_amount'] * claim
    ead_s = np.maximum(ead_s, x['already_drawn_amount'])

    rec_months_s = (
        x['post_claim_recovery_months'] * (1.00 + 0.10 * claim_add - 0.08 * cash_sec_shift)
    ).clip(2.0, 48.0)

    lgd_base_s = (
        0.38 * x['realised_lgd']
        + 0.27 * (1.0 - cash_sec)
        + 0.14 * (1.0 - x['collateral_support_pct'])
        + 0.11 * (1.0 - x['customer_risk_strength'])
        + 0.04 * x['watchlist_flag']
        + 0.04 * ((rec_months_s - 6.0) / 18.0).clip(0.0, 1.0)
        + x['legal_processing_cost_pct']
    ).clip(0.05, 0.92)

    lgd_down_s = (
        lgd_base_s
        * (1.00 + 0.08 * claim + 0.10 * (1.0 - cash_sec) + 0.05 * x['watchlist_flag'])
    ).clip(0.0, 1.0)

    lgd_final_s = np.maximum(
        lgd_down_s + 0.015 + 0.010 * x['watchlist_flag'],
        x['supervisory_floor_proxy'],
    ).clip(0.0, 1.0)

    tmp = x.assign(ead_s=ead_s, lgd_base_s=lgd_base_s, lgd_down_s=lgd_down_s, lgd_final_s=lgd_final_s)
    return {
        'ead_weighted_lgd_base': exposure_weighted_average(tmp, 'lgd_base_s', 'ead_s'),
        'ead_weighted_lgd_downturn': exposure_weighted_average(tmp, 'lgd_down_s', 'ead_s'),
        'ead_weighted_lgd_final': exposure_weighted_average(tmp, 'lgd_final_s', 'ead_s'),
    }

scenarios = [
    {'scenario': 'base', 'claim_add': 0.00, 'cash_sec_shift': 0.00},
    {'scenario': 'claim_prob_+10pp', 'claim_add': 0.10, 'cash_sec_shift': 0.00},
    {'scenario': 'cash_sec_-10pp', 'claim_add': 0.00, 'cash_sec_shift': -0.10},
    {'scenario': 'combined_stress', 'claim_add': 0.10, 'cash_sec_shift': -0.10},
]

sens_rows = []
for s in scenarios:
    m = run_trade_sensitivity(trade, claim_add=s['claim_add'], cash_sec_shift=s['cash_sec_shift'])
    sens_rows.append({'scenario': s['scenario'], **m})

sensitivity = pd.DataFrame(sens_rows)
print('=== Sensitivity Analysis ===')
display(sensitivity.round(4))


In [ ]:
# Monitoring, validation, charts, exports
trade['default_year'] = pd.to_datetime(trade['default_date']).dt.year
monitoring = (
    trade.groupby('default_year', observed=True)
    .apply(
        lambda g: pd.Series(
            {
                'loan_count': len(g),
                'total_ead_base': g['ead_base'].sum(),
                'ead_weighted_lgd_final': exposure_weighted_average(g, 'lgd_final_regulatory', 'ead_base'),
                'mean_recovery_months': g['post_claim_recovery_months'].mean(),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)

validation_checks = pd.DataFrame(
    [
        {'check_name': 'contingent_commitment_non_negative', 'passed': (trade['contingent_commitment_amount'] >= 0).all()},
        {'check_name': 'already_drawn_non_negative', 'passed': (trade['already_drawn_amount'] >= 0).all()},
        {'check_name': 'drawn_not_above_limit', 'passed': (trade['already_drawn_amount'] <= trade['facility_limit_proxy']).all()},
        {'check_name': 'contingent_not_above_undrawn_headroom', 'passed': (trade['contingent_commitment_amount'] <= trade['undrawn_headroom_amount']).all()},
        {'check_name': 'claim_probability_bounds', 'passed': trade['claim_probability_base'].between(0, 1).all()},
        {'check_name': 'ead_base_not_below_drawn', 'passed': (trade['ead_base'] >= trade['already_drawn_amount']).all()},
        {'check_name': 'ead_downturn_not_below_drawn', 'passed': (trade['ead_downturn'] >= trade['already_drawn_amount']).all()},
        {'check_name': 'ead_base_not_above_limit', 'passed': (trade['ead_base'] <= trade['facility_limit_proxy']).all()},
        {'check_name': 'ead_downturn_not_above_limit', 'passed': (trade['ead_downturn'] <= trade['facility_limit_proxy']).all()},
        {'check_name': 'recovery_months_positive', 'passed': (trade['post_claim_recovery_months'] > 0).all()},
        {'check_name': 'downturn_recovery_not_shorter', 'passed': (trade['post_claim_recovery_months_downturn'] >= trade['post_claim_recovery_months']).all()},
    ]
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(weighted_by_product['bucket'], weighted_by_product['ead_weighted_lgd_final'], color='#c44e52')
ax.set_title('Trade / Contingent LGD: Weighted Final LGD by Product Type')
ax.set_xlabel('Product Type')
ax.set_ylabel('EAD-weighted Final LGD')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'trade_contingent_weighted_lgd_by_product.png'), dpi=140, bbox_inches='tight')
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(ead_conversion_summary['security_level'], ead_conversion_summary['ead_weighted_conversion_uplift_pct'], color='#55a868')
ax.set_title('Trade / Contingent EAD Conversion Uplift by Security Level')
ax.set_xlabel('Security Level')
ax.set_ylabel('EAD Conversion Uplift %')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'trade_contingent_ead_conversion_uplift.png'), dpi=140, bbox_inches='tight')
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)

loan_level_output = trade[
    [
        'loan_id', 'facility_type', 'security_type', 'product_type', 'transaction_type_proxy',
        'is_contingent_flag', 'tenor_months', 'industry', 'industry_risk_bucket',
        'facility_limit_proxy', 'undrawn_headroom_amount',
        'contingent_commitment_amount', 'already_drawn_amount', 'claim_probability_base', 'claim_probability_downturn',
        'cash_security_pct', 'collateral_support_pct', 'security_level', 'customer_risk_strength',
        'ead_base', 'ead_downturn', 'ead_conversion_uplift_pct',
        'post_claim_recovery_months', 'post_claim_recovery_months_downturn',
        'legal_processing_cost_pct', 'realised_lgd', 'lgd_base_economic', 'lgd_downturn', 'lgd_final_regulatory',
    ]
].copy()

loan_level_output.to_csv(os.path.join(TABLE_DIR, 'trade_contingent_loan_level_output.csv'), index=False)
segment_summary.to_csv(os.path.join(TABLE_DIR, 'trade_contingent_segment_summary.csv'), index=False)
ead_conversion_summary.to_csv(os.path.join(TABLE_DIR, 'trade_contingent_ead_conversion_summary.csv'), index=False)
base_vs_downturn.to_csv(os.path.join(TABLE_DIR, 'trade_contingent_base_vs_downturn.csv'), index=False)
sensitivity.to_csv(os.path.join(TABLE_DIR, 'trade_contingent_sensitivity.csv'), index=False)
monitoring.to_csv(os.path.join(TABLE_DIR, 'trade_contingent_monitoring_by_year.csv'), index=False)
validation_checks.to_csv(os.path.join(TABLE_DIR, 'trade_contingent_validation_checks.csv'), index=False)

print('=== Validation Checks ===')
display(validation_checks)

print('Saved trade/contingent outputs:')
print('- trade_contingent_loan_level_output.csv')
print('- trade_contingent_segment_summary.csv')
print('- trade_contingent_ead_conversion_summary.csv')
print('- trade_contingent_base_vs_downturn.csv')
print('- trade_contingent_sensitivity.csv')
print('- trade_contingent_monitoring_by_year.csv')
print('- trade_contingent_validation_checks.csv')


---

## APS 113 Calibration Layer — Trade Contingent Facilities

> **SYNTHETIC HISTORICAL CALIBRATION DATA — FOR DEMONSTRATION ONLY**
>
> This section adds a full APS 113-aligned calibration loop on top of the
> proxy baseline above. Workout data is synthetically generated (2014-2024,
> 10-year window) to demonstrate methodology; real production calibration
> requires an internal workout tape.

### Calibration Pipeline (APS 113 s.32-68)

| Step | Function | APS 113 |
|------|----------|---------|
| 1 | Load/generate historical workout data | s.44, Att A |
| 2 | `compute_realised_lgd()` — LIP costs, cure leg | s.32-34 |
| 3 | `classify_economic_regime()` + `assign_regime_to_workouts()` | s.43, s.46 |
| 4 | `segment_lgd()` — product-specific segment keys | s.52 |
| 5 | `compute_long_run_lgd()` — vintage-EWA method | s.43-44 |
| 6 | `compare_model_vs_actual()` — proxy vs realised | s.60-62 |
| 7 | `apply_downturn_overlay()` + `apply_correlation_adjustment()` | s.46-50, s.55-57 |
| 8 | `MoCRegister` + `apply_moc()` — 5 APS 113 s.65 sources | s.63-65 |
| 9 | `apply_regulatory_floor()` — 15% per APS 113 s.58 | s.58 |
| 10 | Export 9 CSV outputs | — |
| 11 | `run_full_validation_suite()` — Gini, HL, PSI, OOT | s.66-68 |

**Correct APS 113 order:** LR-LGD → downturn overlay → correlation adj →
MoC → floor (MoC applied to downturn LGD, not long-run LGD, per s.63).


In [ ]:
# APS 113 Calibration Layer — imports and configuration
import os, sys
from pathlib import Path
sys.path.insert(0, os.path.abspath('..'))

from src.calibration_utils import (
    compute_realised_lgd,
    segment_lgd,
    compute_long_run_lgd,
    compare_model_vs_actual,
    classify_economic_regime,
    assign_regime_to_workouts,
    apply_downturn_overlay,
    apply_correlation_adjustment,
    build_lgd_pd_annual_series,
    estimate_lgd_pd_correlation,
    MoCRegister,
    apply_moc,
    apply_regulatory_floor,
    run_full_validation_suite,
    generate_compliance_map,
    export_compliance_map,
    CALIBRATION_STEP_ORDER,
)
from src.generators import GENERATOR_MAP, generate_all_historical_workouts

PRODUCT = "trade_contingent"
SEGMENT_KEYS = ['facility_type', 'cash_collateral_band']
MODEL_LGD_COL = "lgd_final"

HISTORY_DIR = Path('..') / 'data' / 'generated' / 'historical'
OUTPUTS_DIR = Path('..') / 'outputs' / 'tables'
UPSTREAM_PARQUET = Path('..') / 'data' / 'exports' / 'macro_regime_flags.parquet'

HISTORY_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Product: {PRODUCT}")
print(f"Segment keys: {SEGMENT_KEYS}")
print(f"APS 113 calibration pipeline — step order:")
for step, fn, ref in CALIBRATION_STEP_ORDER:
    print(f"  Step {step:>2}: {fn:<35} {ref}")


In [ ]:
# ── STEP 1: Load or generate historical workout data (APS 113 s.44, Att A) ─
parquet_path = HISTORY_DIR / f"{PRODUCT}_workouts.parquet"

if parquet_path.exists():
    cal_loans = pd.read_parquet(parquet_path)
    # cashflows stored alongside or re-generated
    try:
        cal_cashflows = pd.read_parquet(
            HISTORY_DIR / f"{PRODUCT}_cashflows.parquet"
        )
    except FileNotFoundError:
        cal_cashflows = None
    print(f"Loaded {len(cal_loans):,} historical workout loans from Parquet")
else:
    print(f"Parquet not found — generating synthetic workout data for {PRODUCT}")
    results = generate_all_historical_workouts(
        seed=42, output_dir=HISTORY_DIR, write_parquet=True, products=[PRODUCT]
    )
    cal_loans = results[PRODUCT]["loans"]
    cal_cashflows = results[PRODUCT].get("cashflows")
    print(f"Generated {len(cal_loans):,} synthetic workout loans (2014-2024)")

print(f"Date range: {cal_loans['default_date'].min()} to {cal_loans['default_date'].max()}")
print(f"Columns: {list(cal_loans.columns)}")

# ── STEP 2: Compute realised LGD (APS 113 s.32-34) ─────────────────────────
# LIP costs (Loss Identification Period, ~90 days) auto-detected if cashflows available
lgd_df = compute_realised_lgd(
    loans=cal_loans,
    cashflows=cal_cashflows,
    ead_col="ead_at_default",
    recovery_col="recovery_amount",
    cost_col="direct_costs",
    lip_window_days=90,
)
print(f"\nStep 2: Realised LGD computed")
print(f"  EAD-weighted realised LGD: {(lgd_df['realised_lgd'] * lgd_df['ead_at_default']).sum() / lgd_df['ead_at_default'].sum():.2%}")
display(lgd_df[['realised_lgd', 'ead_at_default']].describe().round(4))

# ── STEP 3: Economic regime classification (APS 113 s.43, s.46-50) ─────────
upstream_path = str(UPSTREAM_PARQUET) if UPSTREAM_PARQUET.exists() else None
regime_df = classify_economic_regime(
    upstream_parquet_path=upstream_path,
    method="upstream_first",
)
print(f"\nStep 3: Economic regimes classified")
print(f"  Data source: {regime_df['data_source'].iloc[0]}")
display(regime_df[['year', 'regime', 'is_downturn_period', 'data_source']].head(12))

lgd_df = assign_regime_to_workouts(lgd_df, regime_df)
downturn_pct = lgd_df.get('is_downturn_period', pd.Series([False])).mean()
print(f"  Downturn observations: {downturn_pct:.1%} of portfolio")

# ── STEP 4: Segment by product-specific keys (APS 113 s.52) ────────────────
segmented_df = segment_lgd(lgd_df, SEGMENT_KEYS)
low_count = segmented_df[segmented_df.get('segment_flag', '') == 'low_count'] if 'segment_flag' in segmented_df.columns else pd.DataFrame()
print(f"\nStep 4: Segmentation complete")
print(f"  Segments: {segmented_df.groupby(SEGMENT_KEYS, observed=True).ngroups}")
if not low_count.empty:
    print(f"  Low-count segments flagged (<20 obs): {len(low_count)}")

# ── STEP 5: Long-run LGD — vintage-EWA (APS 113 s.43-44) ─────────────────
lr_lgd_df = compute_long_run_lgd(
    segmented_df,
    segment_keys=SEGMENT_KEYS,
    method="vintage_ewa",
)
print(f"\nStep 5: Long-run LGD by segment (vintage-EWA)")
display(lr_lgd_df.round(4))
lr_lgd_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_long_run_lgd_by_segment.csv", index=False)

# ── STEP 6: Compare model vs actual (APS 113 s.60-62) ──────────────────────
# 'model_lgd' here = proxy model LGD from the section above (lgd_final if present)
if MODEL_LGD_COL in cal_loans.columns:
    compare_input = lgd_df.merge(
        cal_loans[['loan_id', MODEL_LGD_COL] if 'loan_id' in cal_loans.columns else [MODEL_LGD_COL]],
        left_index=True, right_index=True, how='left',
    ) if MODEL_LGD_COL not in lgd_df.columns else lgd_df
else:
    compare_input = lgd_df.copy()
    compare_input['model_lgd'] = lr_lgd_df['long_run_lgd'].mean() if 'long_run_lgd' in lr_lgd_df.columns else 0.25

model_col_to_use = MODEL_LGD_COL if MODEL_LGD_COL in compare_input.columns else 'model_lgd'
mva_df = compare_model_vs_actual(
    compare_input,
    model_lgd_col=model_col_to_use,
    segment_keys=SEGMENT_KEYS,
)
print(f"\nStep 6: Model vs actual comparison")
display(mva_df.round(4))
mva_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_model_vs_actual_comparison.csv", index=False)

# ── STEP 7: Downturn overlay + Frye-Jacobs correlation adj (s.46-50, s.55-57)
# Reuses apply_downturn_overlay from existing lgd_calculation.py
dt_lgd = apply_downturn_overlay(segmented_df, product=PRODUCT)
print(f"\nStep 7: Downturn overlay applied")
if 'lgd_downturn' in dt_lgd.columns:
    ewa_dt = (dt_lgd['lgd_downturn'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
    ewa_lr = (dt_lgd['realised_lgd'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
    print(f"  EWA Long-run LGD:  {ewa_lr:.2%}")
    print(f"  EWA Downturn LGD:  {ewa_dt:.2%}")
    downturn_col = 'lgd_downturn'
else:
    dt_lgd['lgd_downturn'] = dt_lgd['realised_lgd'] * 1.15  # fallback scalar
    downturn_col = 'lgd_downturn'

# Frye-Jacobs correlation adjustment (APS 113 s.55-57)
try:
    lgd_ts, pd_ts = build_lgd_pd_annual_series(dt_lgd)
    macro_for_corr = regime_df.rename(columns={'gdp_growth': 'gdp_growth_yoy'})
    corr_result = estimate_lgd_pd_correlation(lgd_ts, pd_ts, macro_for_corr)
    dt_lgd['lgd_downturn_corr_adj'] = apply_correlation_adjustment(
        dt_lgd[downturn_col], corr_result['rho'], corr_result['macro_shock_std']
    )
    downturn_col = 'lgd_downturn_corr_adj'
    print(f"  Frye-Jacobs rho={corr_result['rho']:.3f}, adj factor={corr_result['lgd_dt_adjustment_factor']:.3f}")
except Exception as exc:
    print(f"  Frye-Jacobs skipped: {exc}")

dt_lgd.to_csv(OUTPUTS_DIR / f"{PRODUCT}_downturn_lgd_by_segment.csv", index=False)

# ── STEP 8: MoC register + apply MoC (AFTER downturn — APS 113 s.63-65) ───
# Determine regime data source for MoC model_approximation component
data_src = regime_df['data_source'].iloc[0] if 'data_source' in regime_df.columns else 'synthetic'
n_downturn_yrs = int(regime_df['is_downturn_period'].sum()) if 'is_downturn_period' in regime_df.columns else 0

psi_approx = 0.05  # placeholder — full PSI computed in Step 11
bias_approx = float(mva_df['bias'].abs().mean()) if 'bias' in mva_df.columns else 0.02

moc_register = MoCRegister(product=PRODUCT, regime_data_source=data_src)
moc_df = moc_register.build_moc_register(
    segment_df=segmented_df,
    segment_keys=SEGMENT_KEYS,
    n_downturn_vintages=n_downturn_yrs,
    psi_value=psi_approx,
    backtesting_bias=bias_approx,
)
print(f"\nStep 8: MoC register built")
display(moc_df.round(4))
moc_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_moc_register.csv", index=False)

lgd_with_moc = apply_moc(dt_lgd[downturn_col], moc_df, segment_col=SEGMENT_KEYS[0] if SEGMENT_KEYS else None)
dt_lgd['lgd_with_moc'] = lgd_with_moc
moc_ewa = (lgd_with_moc * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
print(f"  EWA LGD after MoC: {moc_ewa:.2%}")

# ── STEP 9: Regulatory floors (APS 113 s.58) ──────────────────────────────
dt_lgd['lgd_final_calibrated'] = apply_regulatory_floor(dt_lgd['lgd_with_moc'], product=PRODUCT)
final_ewa = (dt_lgd['lgd_final_calibrated'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
floor_binding_pct = (dt_lgd['lgd_final_calibrated'] > dt_lgd['lgd_with_moc']).mean()
print(f"\nStep 9: Regulatory floor applied")
print(f"  EWA Final Calibrated LGD: {final_ewa:.2%}")
print(f"  Floor binding for: {floor_binding_pct:.1%} of loans")

dt_lgd.to_csv(OUTPUTS_DIR / f"{PRODUCT}_final_calibrated_lgd.csv", index=False)

# ── STEP 10: Export all outputs ────────────────────────────────────────────
# Already exported: long_run_lgd_by_segment, model_vs_actual, downturn_lgd, moc_register, final_calibrated_lgd
# Export remaining:
lgd_df[['realised_lgd', 'ead_at_default']].assign(product=PRODUCT).to_csv(
    OUTPUTS_DIR / f"{PRODUCT}_historical_workouts.csv", index=False
)
regime_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_regime_classification.csv", index=False)

# Calibration adjustments summary
cal_adj_summary = pd.DataFrame({
    'product': [PRODUCT],
    'ewa_realised_lgd': [(lgd_df['realised_lgd'] * lgd_df['ead_at_default']).sum() / lgd_df['ead_at_default'].sum()],
    'ewa_long_run_lgd': [lr_lgd_df['long_run_lgd'].mean() if 'long_run_lgd' in lr_lgd_df.columns else None],
    'ewa_downturn_lgd': [(dt_lgd.get('lgd_downturn', dt_lgd['realised_lgd']) * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()],
    'ewa_lgd_with_moc': [(dt_lgd['lgd_with_moc'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()],
    'ewa_lgd_final': [final_ewa],
    'floor_binding_pct': [floor_binding_pct],
    'regime_data_source': [data_src],
    'n_loans': [len(lgd_df)],
    'calibration_date': [pd.Timestamp.today().date()],
})
cal_adj_summary.to_csv(OUTPUTS_DIR / f"{PRODUCT}_calibration_adjustments.csv", index=False)
print(f"\nStep 10: All outputs exported to {OUTPUTS_DIR}")

# ── STEP 11: Full validation suite (APS 113 s.66-68) ──────────────────────
print(f"\nStep 11: Running full validation suite...")
try:
    val_results = run_full_validation_suite(
        loans=dt_lgd,
        predicted_col='lgd_final_calibrated',
        actual_col='realised_lgd',
        segment_col=SEGMENT_KEYS[0] if SEGMENT_KEYS else None,
        date_col='default_date' if 'default_date' in dt_lgd.columns else None,
        product=PRODUCT,
    )
    print(f"  Gini: {val_results.get('gini', 'n/a')}")
    print(f"  Calibration ratio: {val_results.get('calibration_ratio', 'n/a')}")
    print(f"  PSI: {val_results.get('psi', 'n/a')}")
    if 'summary_table' in val_results:
        val_results['summary_table'].to_csv(
            OUTPUTS_DIR / f"{PRODUCT}_validation_report.csv", index=False
        )
    if 'backtest_results' in val_results:
        val_results['backtest_results'].to_csv(
            OUTPUTS_DIR / f"{PRODUCT}_backtest_results.csv", index=False
        )
except Exception as exc:
    print(f"  Validation suite error (non-fatal): {exc}")


In [ ]:
# ── Calibration summary waterfall ──────────────────────────────────────────
import matplotlib.pyplot as plt

try:
    stages = {
        'Realised LGD\n(2014-2024)': (lgd_df['realised_lgd'] * lgd_df['ead_at_default']).sum() / lgd_df['ead_at_default'].sum(),
        'Long-run LGD\n(vintage-EWA)': lr_lgd_df['long_run_lgd'].mean() if 'long_run_lgd' in lr_lgd_df.columns else None,
        'Downturn LGD': (dt_lgd.get(downturn_col, dt_lgd['realised_lgd']) * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum(),
        '+ MoC': (dt_lgd['lgd_with_moc'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum(),
        'Final\n(Floor Applied)': final_ewa,
    }
    labels = [k for k, v in stages.items() if v is not None]
    values = [v for v in stages.values() if v is not None]
    colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c', '#8e44ad'][:len(labels)]

    fig, ax = plt.subplots(figsize=(11, 5))
    bars = ax.bar(labels, values, color=colors, edgecolor='white', width=0.6)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.1%}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    ax.set_ylabel('EAD-Weighted LGD')
    ax.set_title(f'APS 113 Calibration Waterfall — Trade Contingent Facilities')
    ax.set_ylim(0, max(values) * 1.35)
    ax.axhline(values[-1], color='black', ls=':', lw=1, label=f'Final: {values[-1]:.1%}')
    ax.legend()
    plt.tight_layout()
    plt.savefig(
        Path('..') / 'outputs' / 'figures' / f'trade_contingent_calibration_waterfall.png',
        dpi=150, bbox_inches='tight',
    )
    plt.show()
except Exception as exc:
    print(f"Waterfall chart error (non-fatal): {exc}")

# ── APS 113 compliance snapshot ─────────────────────────────────────────────
compliance_df = generate_compliance_map(
    calibration_results={PRODUCT: {'long_run_lgd_by_segment': True, 'calibration_steps': True}},
    moc_registers={PRODUCT: moc_df},
    regime_data_source=data_src,
    products=[PRODUCT],
)
print("\n=== APS 113 Compliance Snapshot ===")
display(compliance_df[['section_ref', 'requirement', 'status', 'reviewer_note']].set_index('section_ref'))
export_compliance_map(compliance_df, OUTPUTS_DIR / f"trade_contingent_aps113_compliance.csv")

# Final summary
print("\n=== Calibration Summary ===")
display(cal_adj_summary.round(4))
print(f"\nAll calibration outputs in: {OUTPUTS_DIR}")
print(f"SYNTHETIC HISTORICAL CALIBRATION DATA — FOR DEMONSTRATION ONLY")
